# Hunt: low-and-slow password spray (T1110.003)

**Hypothesis.** One source IP tries a single password against many accounts, slowly enough that the alert on
100 failures in 5 minutes never fires.

1. `params`: the suspect IP from the hot-index triage.
2. `raw-search`: its full history in the Machine Data Lake raw table. The hot index only keeps 14 days.
3. `load-shared`: the shared 90-day analytics table, loaded into this kernel with pandas.
4. `backtest`: the detection rule over 90 days, and the false positives it must skip.
5. `enrich`, `detection`: threat intel and privileged accounts, then the SPL to save as the alert.

In [1]:
SUSPECT_IP = "198.51.100.22"            # from the hot-index triage
WINDOW_DAYS = 90
MIN_USERS = 40             # distinct accounts failed per IP per day
MAX_TRIES_PER_USER = 3     # low and slow: a few attempts per account at most
MIN_FAIL_RATIO = 0.8       # a spray fails almost every attempt

In [2]:
import pandas as pd
import plotext as plt
import lake

print(lake.describe())

lake: live | raw demo_auth_raw | shared demo_auth_hunt_90d


In [3]:
raw = lake.search_raw(SUSPECT_IP, days=120)
weekly = raw.set_index("_time").resample("W")["user"].size()
print(f"{len(raw):,} events from {SUSPECT_IP}, "
      f"{raw['user'].nunique()} accounts, "
      f"first seen {raw['_time'].min():%b %d}")
plt.clear_figure(); plt.theme("clear"); plt.plot_size(76, 9)
plt.bar([f"{d:%m-%d}" for d in weekly.index], list(weekly))
plt.title("events per week from the suspect IP")
plt.show()

679 events from 198.51.100.22, 335 accounts, first seen Sep 07
                      events per week from the suspect IP                   
   ┌───────────────────────────────────────────────────────────────────────┐
372┤                                                           ██████      │
310┤                                                           ██████      │
186┤██████                                                     ██████      │
124┤██████                                               ██████████████████│
  0┤██████                                               ██████████████████│
   └──┬─────┬─────┬─────┬─────┬─────┬─────┬───────────┬─────┬───────────┬──┘
   09-13  09-20 09-27 10-04 10-11 10-18 10-25      11-08  11-15      11-29  


In [4]:
tbl = lake.load_shared()
tbl["day"] = tbl["_time"].dt.floor("D")
tbl["is_fail"] = tbl["outcome"].eq("fail")
print(f"{len(tbl):,} sign-ins, {tbl['src_ip'].nunique()} source IPs, "
      f"{tbl['day'].min():%b %d} to {tbl['day'].max():%b %d}")
print(f"{tbl.memory_usage(deep=True).sum() / 1e6:.1f} MB in memory")

104,198 sign-ins, 405 source IPs, Aug 26 to Nov 24
8.2 MB in memory


In [5]:
g = ["src_ip", "day"]
daily = tbl.groupby(g).agg(attempts=("user", "size"),
                           fails=("is_fail", "sum"))
failed = tbl[tbl["is_fail"]]
daily["users"] = failed.groupby(g)["user"].nunique()
daily["users"] = daily["users"].fillna(0)
daily["ratio"] = daily["fails"] / daily["attempts"]
daily["tries"] = daily["fails"] / daily["users"].clip(lower=1)
rule = ((daily["users"] >= MIN_USERS)
        & (daily["ratio"] >= MIN_FAIL_RATIO)
        & (daily["tries"] <= MAX_TRIES_PER_USER))
hits = daily[rule].reset_index()
flagged = hits.groupby("src_ip").agg(days=("day", "size"),
                                     first=("day", "min"),
                                     last=("day", "max"))
print(f"rule flags {len(flagged)} IP in {WINDOW_DAYS} days:")
print(flagged.assign(first=flagged["first"].dt.strftime("%b %d"),
                     last=flagged["last"].dt.strftime("%b %d")))
five = failed.groupby(["src_ip", failed["_time"].dt.floor("5min")])
print(f"old alert: busiest 5 min is {five.size().max()} "
      "failures, it needs 100")
busy = daily[(daily["users"] >= MIN_USERS) & ~rule]
print(f"skipped: {busy.reset_index()['src_ip'].nunique()} busy IP, "
      f"failure ratio at most {busy['ratio'].max():.0%}")
days = pd.date_range(tbl["day"].min(), tbl["day"].max(), freq="D")
series = daily.xs(SUSPECT_IP, level="src_ip")["users"]
series = series.reindex(days, fill_value=0)
plt.clear_figure(); plt.theme("clear"); plt.plot_size(76, 9)
plt.bar([f"{d:%m-%d}" for d in series.index], list(series))
plt.hline(MIN_USERS)
plt.title("accounts failed per day by the suspect IP")
plt.show()

rule flags 1 IP in 90 days:
               days   first    last
src_ip                             
198.51.100.22    13  Sep 07  Nov 24
old alert: busiest 5 min is 6 failures, it needs 100
skipped: 1 busy IP, failure ratio at most 10%
                    accounts failed per day by the suspect IP               
    ┌──────────────────────────────────────────────────────────────────────┐
58.0┤          ██                                                  █ █████ │
48.3├─────────███──────────────────────────────────────────────────████████┤
29.0┤         ███                                                  ████████│
19.3┤         ███                                                  ████████│
 0.0┤         ██                                                   ████████│
    └───┬─────┬───────┬──────────┬────┬─────────┬──────┬─────┬────────────┬┘
     08-30  09-07   09-17      10-01 10-08    10-21  10-31 11-07      11-24 


In [6]:
intel = pd.read_csv("threat_intel.csv")
priv = pd.read_csv("privileged_users.csv")
ips = list(flagged.index)
print(intel[intel["indicator"].isin(ips)].to_string(index=False))
ok = tbl[tbl["src_ip"].isin(ips) & ~tbl["is_fail"]]
wins = ok[["_time", "src_ip", "user"]].merge(priv, on="user")
wins["_time"] = wins["_time"].dt.strftime("%b %d %H:%M")
print(f"privileged accounts the suspect got into: {len(wins)}")
print(wins.to_string(index=False))

    indicator          category first_reported         source
198.51.100.22 password spraying     2026-09-04 community feed
privileged accounts the suspect got into: 1
       _time        src_ip       user            role
Nov 22 03:12 198.51.100.22 svc_backup backup operator


In [7]:
spl = f"""index=demo_auth sourcetype="demo:auth:lake"
| eval failed=if(outcome="fail", 1, 0)
| stats count AS attempts, sum(failed) AS fails,
    dc(eval(if(failed=1, user, null()))) AS users BY src_ip
| eval ratio=fails/attempts, tries=fails/users
| where users>={MIN_USERS} AND ratio>={MIN_FAIL_RATIO}
    AND tries<={MAX_TRIES_PER_USER}"""
print("Daily saved search, ready for review:")
print(spl)

Daily saved search, ready for review:
index=demo_auth sourcetype="demo:auth:lake"
| eval failed=if(outcome="fail", 1, 0)
| stats count AS attempts, sum(failed) AS fails,
    dc(eval(if(failed=1, user, null()))) AS users BY src_ip
| eval ratio=fails/attempts, tries=fails/users
| where users>=40 AND ratio>=0.8
    AND tries<=3
